In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import logging

logging.basicConfig(level=logging.DEBUG)

# prepare data

train_path = '/content/drive/MyDrive/Subtask_A/subtaskA_train_multilingual.jsonl' # train_path = '/content/drive/MyDrive/Subtask_A/subtaskA_train_monolingual.jsonl'
test_path = '/content/drive/MyDrive/Subtask_A/subtaskA_dev_multilingual.jsonl' # test_path = '/content/drive/MyDrive/Subtask_A/subtaskA_dev_monolingual.jsonl'

train_df = pd.read_json(train_path, lines=True)
test_df = pd.read_json(test_path, lines=True)

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

In [ ]:
# run this after restarting colab
!pip install evaluate
!pip install hf_xet


In [ ]:
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, AutoTokenizer

# pandas dataframe to huggingface Dataset
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(val_df)

def preprocess_function(examples, **fn_kwargs):
    return fn_kwargs['tokenizer'](examples["text"], truncation=True)

# model and labels
model = 'xlm-roberta-base' # 'roberta-base'
id2label = {0: "human", 1: "machine"}
label2id = {"human": 0, "machine": 1}

# get tokenizer and model from huggingface
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForSequenceClassification.from_pretrained(model, num_labels=len(label2id), id2label=id2label, label2id=label2id)

# tokenize data for train/valid
tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True, fn_kwargs={'tokenizer': tokenizer})
tokenized_valid_dataset = valid_dataset.map(preprocess_function, batched=True,  fn_kwargs={'tokenizer': tokenizer})


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
import os
import numpy as np
import evaluate
from peft import LoraConfig, TaskType, get_peft_model

def compute_metrics(eval_pred):

    f1_metric = evaluate.load("f1")

    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    results = {}
    results.update(f1_metric.compute(predictions=predictions, references = labels, average="micro"))

    return results

checkpoints_path = '/content/drive/MyDrive/Subtask_A/checkpoints'

if not os.path.exists(checkpoints_path):
    os.makedirs(checkpoints_path)

lora_r = 64
lora_alpha = 16
lora_dropout = 0.1

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=["query", "value"],
)

# model.gradient_checkpointing_enable()
# model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

# create Trainer
training_args = TrainingArguments(
    output_dir=checkpoints_path,
    learning_rate=2e-4, # 2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_valid_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting to train model")
trainer.train()

# save best model
best_model_path = checkpoints_path+'/best_xlm_roberta/' # checkpoints_path+'/best_roberta/'

if not os.path.exists(best_model_path):
    os.makedirs(best_model_path)


trainer.save_model(best_model_path)

In [ ]:
# Evaluierung des Modells
from scipy.special import softmax
import evaluate
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, AutoTokenizer

def preprocess_function(examples, **fn_kwargs):
    return fn_kwargs['tokenizer'](examples["text"], truncation=True)

def compute_metrics(eval_pred):

    f1_metric = evaluate.load("f1")

    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    results = {}
    results.update(f1_metric.compute(predictions=predictions, references = labels, average="micro"))

    return results

def test(test_df, model_path, id2label, label2id):

    # load tokenizer from saved model
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    # load best model
    model = AutoModelForSequenceClassification.from_pretrained(
       model_path, num_labels=len(label2id), id2label=id2label, label2id=label2id
    )

    test_dataset = Dataset.from_pandas(test_df)

    tokenized_test_dataset = test_dataset.map(preprocess_function, batched=True,  fn_kwargs={'tokenizer': tokenizer})
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # create Trainer
    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    # get logits from predictions and evaluate results using classification report
    predictions = trainer.predict(tokenized_test_dataset)
    prob_pred = softmax(predictions.predictions, axis=-1)
    preds = np.argmax(predictions.predictions, axis=-1)
    metric = evaluate.load("bstrai/classification_report")
    results = metric.compute(predictions=preds, references=predictions.label_ids)

    return results, preds


model_path = "/content/drive/MyDrive/Subtask_A/checkpoints/best_xlm_roberta" # "/content/drive/MyDrive/Subtask_A/checkpoints/best_roberta"
id2label = {0: "human", 1: "machine"}
label2id = {"human": 0, "machine": 1}

test_path = '/content/drive/MyDrive/Subtask_A/subtaskA_dev_multilingual.jsonl' # '/content/drive/MyDrive/Subtask_A/subtaskA_dev_monolingual.jsonl'

test_df = pd.read_json(test_path, lines=True)

results, preds = test(test_df, model_path, id2label, label2id)

print(results)

